In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io.wavfile import write


Learning Outcome: Understand the components that build the FFT including the real and imaginary parts. The students will plot both parts and observe the symmetry of the FFT, while finally plotting the one sided real part of the FFT. The questions aim to calrify the understanding of the output of the FFT which desribes the frequency of sinusoids the original signal is composed of.

In [ ]:


# Edit these parameters if you would like

Fs = 1000      # sampling rate Hz
T  = 1.0       # duration s
f1, f2 = 50, 120
A1, A2 = 1.0, 0.5
noise_std = 0.0


def make_signal(Fs=1000, T=1.0, f1=50, f2=120, A1=1.0, A2=0.5, noise_std=0.0, seed=None):
    """
    Create a 2-tone sinusoidal signal with optional Gaussian noise.

    Returns:
        t : time array [s] with samples over [0, T)
        x : signal array
        N : number of samples
    """
    # TODO 1: compute number of samples N from Fs and T 

    # TODO 2: build time vector t of length N over [0, T)

    # Optional noise generator for more realistic signal
    rng = np.random.default_rng(seed)

    # TODO 3: construct 2-tone signal:
    # x(t) = A1*sin(2*pi*f1*t) + A2*sin(2*pi*f2*t) + Gaussian noise
    # Hint: rng.standard_normal(N) gives N(0,1) noise; scale by noise_std

    return t, x, N


# Generate a signal 
t, x, N = make_signal(Fs=Fs, T=T, f1=f1, f2=f2, A1=A1, A2=A2, noise_std=noise_std, seed=0)

# Quick sanity checks 
dt = 1 / Fs
df = Fs / N
print(f"N = {N}, dt = {dt:.6f} s, df = {df:.3f} Hz")


# FFT computation
# TODO 4: compute the FFT of x 
#

# TODO 5: build the frequency axis in Hz to freqs



# TODO 6: Plot both real and imaginary parts vs frequency


# TODO 7: Plot only real part vs frequency


# One-sided magnitude plot
def plot_spectrum(X, Fs, title="One-sided magnitude spectrum"):
    """
    Plot a one-sided magnitude spectrum 

    """
    N = X.size

    # TODO 8: build frequency axis in Hz using np.fft.fftfreq

    # TODO 9: mask non-negative frequencies

    # TODO 10: compute magnitude 

    plt.figure(figsize=(8, 3))
    plt.plot(freqs[mask], mag[mask])
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Amplitude")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()



# plot_spectrum(X, Fs, title="FFT one-sided magnitude")

1) Explain what the real and imaginary parts represent physically

2) Why does the FFT of a real-valued signal produce a symmetric pattern around 0 Hz?

3) Identify the frequencies of the main peaks and explain why they occur there.

4) Why do we divide by (N/2) when plotting the magnitude? What does this scaling represent?


Learning Outcome: Allows the student to understand the difference between the DFT and the FFT by manually implementing the DFT. From this the student will observe the computation expense of the DFT and why the FFT was developed. The questions aim to dig into the intuition of the Fourier Transform rather than giving the student a surface level understanding by calling np.fft.fft(X).

In [ ]:

# DFT matrix 
def dft_matrix(N):
    """
    Build the NxN DFT matrix:
        W[k, n] = exp(-j 2 pi k n / N)
    """
    # TODO: create index arrays k (Nx1) and n (1xN)

    # TODO: fill in the exponential for W (use 1j for j = sqrt(-1))
    return W


# Manual DFT          
def manual_dft(x):
    """
    Compute the DFT via matrix-vector multiply:
        X = W @ x
    """
    N = x.size
    W = dft_matrix(N)
    # TODO: perform the matrix-vector multiply
    return X, W


def show_dft_matrix(W):
    """
    Visualize Re{W} and Im{W} as heatmaps. Red=positive, Blue=negative.
    """
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(np.real(W), aspect="auto", cmap="bwr", interpolation="nearest")
    plt.title("Re{W}")
    plt.colorbar()
    plt.subplot(1, 2, 2)
    plt.imshow(np.imag(W), aspect="auto", cmap="bwr", interpolation="nearest")
    plt.title("Im{W}")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


# TODO: choose parameters 
Fs = 1000    
T  = 1.0       
f1 = 50        
f2 = 120       
A1, A2 = 1.0, 0.5
noise = 0.0

# Build signal
t, x, N = make_signal(Fs=Fs, T=T, f1=f1, f2=f2, A1=A1, A2=A2, noise=noise)

# Manual DFT
X_manual, W = manual_dft(x)
plot_spectrum(X_manual, Fs, title="Manual DFT Magnitude")
show_dft_matrix(W)

# TODO: compute FFT of x and compare with manual DFT

# TODO: Plot the FFT now and compare this against your manual implementation


Look at the heatmaps of the real and imaginary parts of the DFT matrix W.

1) What do the colour patterns along each row represent in terms of sinusoidal frequency?

2) Explain why the DFT matrix can be interpreted as a set of basis sinusoids for analysing the signal.

3) Explain why the FFT produces the same result as the manual DFT but runs much faster.

4) How does changing the number of samples or sampling rate affect the resolution and spacing of frequency bins?

Learning Outcome: The student will now create a simple filter which will mask high frequencies. This will allow the understanding of transforming the signal from the time domain into the frequency domain which will then will be altered here. Once altered the signal will then be transformed back into the time domain by taking the inverse FFT. The reconstruction will be analysed using the signal noise ratio.

In [ ]:
def make_signal(Fs, T, f1=80, f2=180, A1=1.0, A2=0.7, f_noise=600, A_noise=1.0, noise_std=0.2):
    N = int(Fs * T)
    t = np.arange(N) / Fs
    x_clean = A1*np.sin(2*np.pi*f1*t) + A2*np.sin(2*np.pi*f2*t)
    pitched = A_noise*np.sin(2*np.pi*f_noise*t)
    x = x_clean + pitched + noise_std*np.random.randn(N)
    return t, x_clean, x, N

Fs = 2000
T  = 1.0

f1, f2 = 80, 180
A1, A2 = 1.0, 0.7

f_noise   = 600     # pitched interference
A_noise   = 1.0
noise_std = 0.25

notch_bw = 5.0      # half-width of notch in Hz 


rng = np.random.default_rng(0)
np.random.seed(0)  

t, x_clean, x, N = make_signal(Fs, T, f1, f2, A1, A2, f_noise, A_noise, noise_std)


X = np.fft.rfft(x)
freqs = np.fft.rfftfreq(N, d=1/Fs)


mask = np.ones_like(freqs, dtype=float)

# TODO implement notch: set mask=0 inside [f_noise - notch_bw, f_noise + notch_bw]

# TODO Inverse FFT back to time

# TODO Plots: time domain

# TODO Plots: frequency domain before/after + show mask

#TODO write a function which compares the bandpower and then reports the suppression within the notched area



Inspect the plots of the noisy and denoised signals in both time and frequency domains.

1) What effect does the low-pass cutoff f_cut have on the signal’s spectrum?

2) How does this relate to the suppresion of the bandpower and what does the suppresion in dB represent?

3) In your FFT-based notch, the frequency bin spacing is $\Delta f = \frac{F_s}{N}$.
(a) Explain how $\Delta f$ limits how precisely you can target a narrow interference frequency.
(b) If the interference frequency is not exactly centered on an FFT bin, why does spectral leakage occur, and how would windowing change the leakage and the notch design?


Learning Outcomes: Demonstrate how the FFT can be used to compute derivatives and allow the student to understand why the Fourier Transform proivides a more accurate derivative compared to the finite difference method by expanding the signal in global sinusoids. This should also provide the student with an understanding of the power of the Fourier Transform. 

In [ ]:

n = 15      # number of grid points
L = 30         # domain length
dx = L / n     # grid spacing
x = np.linspace(-L/2, L/2 - dx, n)  # uniform grid

#Function and true derivative
f = np.cos(x) * np.exp(-x**2 / 25)
df_true = ( -np.sin(x) * np.exp(-x**2 / 25) + ( -2*x/25 ) * np.cos(x) * np.exp(-x**2 / 25) )

# TODO: implement finite difference method for computing a derivative
#Finite difference derivative 


# TODO: Compute the derivative using fft
# FFT derivative

# TODO: Inverse fourier transform to return this back into physical space

# TODO: Plot results

# TODO: Plot error metrics



Look at the plotted results of the true derivative, the finite-difference derivative, and the FFT derivative.

1) Which method gives a closer match to the true derivative (try changing N to see this effect more clearly)?

2) Explain why this method performs better, based on how each approach computes derivatives.
